Import classes and libraries

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, MapType, FloatType

Define the schema

In [0]:
ipl_schema = StructType([
    StructField("meta", StructType([
        StructField("data_version", StringType(), True),
        StructField("created", StringType(), True),
        StructField("revision", IntegerType(), True)
    ]), True),
    StructField("info", StructType([
        StructField("balls_per_over", IntegerType(), True),
        StructField("city", StringType(), True),
        StructField("dates", ArrayType(StringType()), True),
        StructField("event", StructType([
            StructField("match_number", IntegerType(), True),
            StructField("name", StringType(), True)
        ]), True),
        StructField("gender", StringType(), True),
        StructField("match_type", StringType(), True),
        StructField("officials", StructType([
            StructField("match_referees", ArrayType(StringType()), True),
            StructField("reserve_umpires", ArrayType(StringType()), True),
            StructField("tv_umpires", ArrayType(StringType()), True),
            StructField("umpires", ArrayType(StringType()), True)
        ]), True),
        StructField("outcome", StructType([
            StructField("by", StructType([
                StructField("runs", IntegerType(), True)
            ]), True),
            StructField("winner", StringType(), True)
        ]), True),
        StructField("overs", IntegerType(), True),
        StructField("player_of_match", ArrayType(StringType()), True),
        StructField("players", MapType(StringType(), ArrayType(StringType())), True),
        StructField("registry", StructType([
            StructField("people", MapType(StringType(), StringType()), True)
        ]), True),
        StructField("season", StringType(), True),
        StructField("team_type", StringType(), True),
        StructField("teams", ArrayType(StringType()), True),
        StructField("toss", StructType([
            StructField("decision", StringType(), True),
            StructField("winner", StringType(), True)
        ]), True),
        StructField("venue", StringType(), True)
    ]), True),
    StructField("innings", ArrayType(
        StructType([
            StructField("team", StringType(), True),
            StructField("overs", ArrayType(
                StructType([
                    StructField("over", IntegerType(), True),
                    StructField("deliveries", ArrayType(
                        StructType([
                            StructField("batter", StringType(), True),
                            StructField("bowler", StringType(), True),
                            StructField("non_striker", StringType(), True),
                            StructField("extras", StructType([
                                StructField("legbyes", IntegerType(), True),
                                StructField("wides", IntegerType(), True),
                                StructField("byes", IntegerType(), True)
                            ]), True),
                            StructField("runs", StructType([
                                StructField("batter", IntegerType(), True),
                                StructField("extras", IntegerType(), True),
                                StructField("total", IntegerType(), True)
                            ]), True),
                            StructField("wickets", ArrayType(
                                StructType([
                                    StructField("kind", StringType(), True),
                                    StructField("player_out", StringType(), True),
                                    StructField("fielders", ArrayType(
                                        StructType([
                                            StructField("name", StringType(), True)
                                        ])
                                    ), True)
                                ])
                            ), True)
                        ])
                    ), True)
                ])
            ), True),
            StructField("powerplays", ArrayType(
                StructType([
                    StructField("from", DoubleType(), True),
                    StructField("to", DoubleType(), True),
                    StructField("type", StringType(), True)
                ])
            ), True),
            StructField("target", StructType([
                StructField("overs", IntegerType(), True),
                StructField("runs", IntegerType(), True)
            ]), True)
        ])
    ), True)
])

Read Data from Source

In [0]:
source_ipl_json = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format','json') \
    .option('cloudFiles.schemaLocation','/Volumes/workspace/source/sourceipldata/_schemas') \
    .schema(ipl_schema) \
    .option('cloudFiles.schemaEvolutionMode','rescue') \
    .load('/Volumes/workspace/source/sourceipldata/*.json',multiLine=True)



In [0]:
source_ipl_json.writeStream.format('delta') \
    .option('checkpointLocation','/Volumes/workspace/source/sourceipldata/_checkpoints') \
    .outputMode('append') \
    .trigger(once=True) \
    .toTable('ipl.raw.iplData')